<a href="https://colab.research.google.com/github/Shads-GoldenCorsair/sumo-interseccao-eduardo-mondlane-salvador-allende/blob/master/agente_dqn/treino_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treino DQN, interseccao Eduardo Mondlane / Salvador Allende (Google Colab)

Corre o treino completo do agente DQN sem ocupar a maquina local. Ver `PROGRESSO.md` no repositorio para o contexto completo.

**Como usar:**
1. Corre as celulas por ordem.
2. Na celula de clone, cola um *personal access token* do GitHub quando pedido (scope `repo`, porque o repositorio e privado). O token nao fica gravado no notebook.
3. Escolhe `CENARIO` na celula de configuracao. As 5 sementes correm todas seguidas, nesta mesma sessao (nao em paralelo).
4. No fim, a ultima celula faz commit e push do CSV de resultados para o repositorio.
5. Repete tudo para o outro `CENARIO` (pico/baixo_fluxo) noutra sessao.

## 1. Instalar o SUMO

In [1]:
!apt-get update -qq
!apt-get install -y -qq sumo sumo-tools sumo-doc
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"
!sumo --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package binfmt-support.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../00-binfmt-support_2.2.2-7_amd64.deb ...
Unpacking binfmt-support (2.2.2-7) ...
Selecting previously unselected package fonts-roboto-unhinted.
Preparing to unpack .../01-fonts-roboto-unhinted_2%3a0~20170802-3_all.deb ...
Unpacking fonts-roboto-unhinted (2:0~20170802-3) ...
Selecting previously unselected package fastjar.
Preparing to unpack .../02-fastjar_2%3a0.98-7_amd64.deb ...
Unpacking fastjar (2:0.98-7) ...
Selecting previously unselected package jarwrapper.
Preparing to unpack .../03-jarwrapper_0.79_all.deb ...
Unpacking jarwrapper (0.79) ...
Selecting previously unselected package javascript-common.
Preparing to unpack .../04-javascript-common_

## 2. Clonar o repositorio privado

Precisas de um *personal access token* do GitHub (Settings > Developer settings > Fine-grained tokens, scope de leitura/escrita sobre este repositorio, ou um classic token com scope `repo`). Cola-o quando pedido; fica so na memoria desta sessao Colab.

In [2]:
from getpass import getpass

token = getpass("Cola aqui o teu GitHub personal access token: ")
utilizador = "Shads-GoldenCorsair"
repo = "sumo-interseccao-eduardo-mondlane-salvador-allende"

!git clone https://{token}@github.com/{utilizador}/{repo}.git
%cd {repo}
del token  # nao deixa o token na variavel apos clonar

Cola aqui o teu GitHub personal access token: ··········
Cloning into 'sumo-interseccao-eduardo-mondlane-salvador-allende'...
remote: Enumerating objects: 271, done.
remote: Counting objects: 100% (271/271), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 271 (delta 174), reused 205 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (271/271), 3.64 MiB | 4.63 MiB/s, done.
Resolving deltas: 100% (174/174), done.
/content/sumo-interseccao-eduardo-mondlane-salvador-allende


## 3. Instalar dependencias Python

In [3]:
!pip install -q -r agente_dqn/requirements.txt

## 4. Configuracao desta sessao

As 5 sementes correm todas seguidas, nesta mesma sessao (nao em paralelo). `EPISODIOS` e por semente (5 sementes x EPISODIOS = total de episodios simulados).

In [ ]:
CENARIO = "baixo_fluxo"  # ou "pico"
SEMENTES = 5              # corre as sementes 0,1,2,3,4 todas seguidas
EPISODIOS = 100           # por semente, valor de trabalho (ver PROGRESSO.md)

## 4b. Ligar checkpoints ao Google Drive

O `train.py` grava progresso (pesos + episodio actual) a cada 10 episodios de cada semente, para retomar se a sessao for interrompida. Mas o disco do Colab e efemero: se a sessao for reiniciada, o que la estiver perde-se. Esta celula guarda os checkpoints das 5 sementes no teu Google Drive, na mesma pasta (o `train.py` ja separa os ficheiros por semente dentro dela). Vai pedir autorizacao de acesso ao Drive na primeira vez.

Se ja tinhas uma pasta antiga `..._semente0` de uma sessao anterior (antes de mudarmos para todas as sementes numa so sessao), move os 2 ficheiros de la para dentro desta pasta nova no teu Drive, para nao perderes esse progresso.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
pasta_drive = f"/content/drive/MyDrive/pfc_checkpoints_dqn/{CENARIO}"
os.makedirs(pasta_drive, exist_ok=True)

!rm -rf outputs/_checkpoints
!ln -s "{pasta_drive}" outputs/_checkpoints
print("checkpoints desta sessao gravados em:", pasta_drive)

## 5. Treinar

Fica a correr nesta celula durante muito tempo (5 sementes x 100 episodios seguidas). O Colab gratuito desliga por inactividade (~90min sem interaccao) ou ao fim de algumas horas de sessao continua. Se isso acontecer, volta a abrir o notebook e corre as celulas por ordem outra vez (incluindo a 4b): o `train.py` deteta o checkpoint no Drive e retoma a partir da ultima semente/episodio gravado, em vez de recomecar do zero.

In [ ]:
!cd agente_dqn && python -u train.py --cenario {CENARIO} --episodios {EPISODIOS} --sementes {SEMENTES}

## 6. Enviar os resultados de volta para o GitHub

Grava o CSV com as 5 sementes juntas (o `train.py` ja escreve tudo no mesmo ficheiro quando corre varias sementes seguidas).

In [ ]:
!git config user.email "colab@treino.local"
!git config user.name "Treino Colab"
!git add outputs/treino_{CENARIO}.csv
!git commit -m "Resultados treino {CENARIO}, 5 sementes"
!git pull --no-edit --no-rebase origin master
!git push origin master